# Hashable or not? Python dictionary keys
**Python Foundation · Lawson Dong · Notebook 09**

An editable, runnable Google Colab notebook on **hash values, dictionary keys, and hashability**.

**How to use:** Select **Runtime → Run all**, or run each cell. Change the example values and rerun. All intentional errors are caught so *Run all* can finish.

> **Mental model:** A Python object is **hashable** when it has a hash value that stays stable during its lifetime, and objects that compare equal must have equal hashes. Dictionaries and sets rely on this rule to organize their keys and elements.


## 1. A hash is an integer, not a unique ID

Python's built-in `hash(object)` returns an integer for a hashable object. It must remain stable **within that object's lifetime**.

- Equal objects must have equal hashes.
- Different objects **can** have the same hash (a *hash collision*).
- For strings, exact hash numbers can change between Python processes. Do not memorize or hard-code them.


In [ ]:
examples = [42, 3.14, "apple", True, (1, 2)]
for value in examples:
    print(f"{value!r:>12}  type={type(value).__name__:>5}  hash={hash(value)}")

word = "apple"
assert hash(word) == hash(word)
print("\nRepeated hashing of the same string in this run:", hash(word), hash(word))


## 2. Dictionary keys do not have to be strings

### Which built-in types can be dictionary keys?

| Python type | Hashable? | Example as a dictionary key |
|:--|:--:|:--|
| `str` | ✅ Yes | `"name"` |
| `int` | ✅ Yes | `42` |
| `float` | ✅ Yes | `3.14` |
| `bool` | ✅ Yes | `True` |
| `tuple` | ⚠️ Depends | `(1, "x")` ✅, but `(1, [2])` ❌ |
| `list` | ❌ No | `[1, 2]` |
| `dict` | ❌ No | `{"a": 1}` |
| `set` | ❌ No | `{1, 2}` |

**The rule is not “keys must be strings” or even “keys must be immutable.”** A dictionary key must be **hashable**. A `tuple` can be a key **only when every item inside it is hashable**. Conversely, `list`, `dict`, and `set` cannot be keys because these built-in container types are unhashable.

Try a string, an integer, a boolean, and a tuple in the next cell. They all work.

**Important:** Two keys that compare equal and have the same hash act as the same dictionary key; for example, `True` and `1` (explained in Section 5).


In [ ]:
lookup = {
    "name": "Ada",
    42: "answer",
    False: "boolean key",
    (1, 2): "coordinates",
}
for key, value in lookup.items():
    print(f"{key!r:>10} ({type(key).__name__}): {value}")

assert lookup[42] == "answer"
assert lookup[(1, 2)] == "coordinates"


## 3. Test hashability instead of guessing

Usually, `str`, `int`, `float`, `bool`, `None`, and tuples of hashable elements are hashable.

Regular `list`, `dict`, and `set` objects are **not** hashable. A tuple containing a list is also unhashable, **even though the tuple itself cannot be reassigned**.

The code catches `TypeError` instead of stopping the whole notebook.


In [ ]:
# Check ALL eight types in the table — and some tricky extras.
# Each candidate is tried in its own dictionary.
examples = [
    ("str", "hello"),
    ("int", 7),
    ("float", 3.14),
    ("bool", True),
    ("tuple (hashable elements)", (1, "x")),
    ("list", [1, 2]),
    ("dict", {"a": 1}),
    ("set", {1, 2}),
    ("tuple containing a list", (1, [2, 3])),
    ("frozenset (bonus)", frozenset({1, 2})),
]

for label, value in examples:
    try:
        value_hash = hash(value)
        test_dictionary = {value: "success"}
        assert test_dictionary[value] == "success"
        print(f"YES | {label:<28} | hash={value_hash}")
    except TypeError as error:
        print(f"NO  | {label:<28} | {type(error).__name__}: {error}")

# Key idea: a hash value helps the dictionary locate a key, but equality also matters.


## 4. Why mutable containers cannot usually be keys

A list can change after creation. If a key's hash or equality behavior changed while inside a dictionary, finding that key could stop working correctly.

A tuple does not let you replace its elements, but its elements can still include mutable, unhashable objects. **Immutable is not a perfect synonym for hashable.**

Convert a list into a tuple **only when its elements are themselves hashable**. For sets, a `frozenset` is an immutable, hashable alternative when its elements are hashable.


In [ ]:
parts = [1, 2, 3]
parts[0] = 99                    # Lists can change.
print("Modified list:", parts)

safe_key = tuple(parts)
point_data = {safe_key: "stored successfully"}
print("Tuple key:", safe_key, "→", point_data[safe_key])

safe_set_key = frozenset({2, 4, 6})
print("Frozenset key works:", {safe_set_key: "even numbers"})

try:
    bad_key = (1, [2, 3])        # Tuple contains an unhashable list.
    hash(bad_key)
except TypeError as error:
    print("Nested mutable container:", type(error).__name__, "-", error)


## 5. The exam trap: `True`, `1`, and `1.0`

In Python, `True == 1 == 1.0` is `True`, and these three values have the same hash.

A dictionary treats them as **the same key**: assigning a later value replaces the previous value but keeps the original key object.

Dictionary keys use both hash values **and equality**, not hashes alone.


In [ ]:
print("Equal values?", True == 1 == 1.0)
print("Hashes:", hash(True), hash(1), hash(1.0))

d = {1: "one"}
d[True] = "true"
d[1.0] = "one point zero"

print("Dictionary:", d)
print("Number of keys:", len(d))
assert len(d) == 1
assert d[1] == "one point zero"


## 6. Interactive hashability explorer

Select an example, then click **Check key**. The program tries `hash(value)` and uses it as a dictionary key.

You can also edit the `samples` dictionary below to add your own cases. This uses standard `ipywidgets`, which works in Google Colab.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # Works in ordinary Jupyter too.

samples = {
    "String — 'apple'": "apple",
    "Integer — 10": 10,
    "Boolean — True": True,
    "Tuple — (1, 2)": (1, 2),
    "List — [1, 2]": [1, 2],
    "Dict — {'a': 1}": {"a": 1},
    "Set — {1, 2}": {1, 2},
    "Tuple containing list": (1, [2, 3]),
    "Frozen set": frozenset({1, 2}),
}
choice = widgets.Dropdown(options=list(samples), description="Example:", style={"description_width": "initial"})
button = widgets.Button(description="Check key", button_style="info")
result = widgets.Output()

def inspect_key(_=None):
    with result:
        result.clear_output()
        value = samples[choice.value]
        print("Value:", repr(value))
        print("Type:", type(value).__name__)
        try:
            number = hash(value)
            example_dict = {value: "accepted"}
            print("Hash:", number)
            print("Dictionary insertion: SUCCESS", repr(example_dict))
        except TypeError as error:
            print("Dictionary insertion: NOT ALLOWED")
            print(type(error).__name__ + ":", error)

button.on_click(inspect_key)
display(widgets.VBox([choice, button, result]))
inspect_key()


## 7. A nuance: hash collisions do not mean equal keys (optional)

A *collision* occurs when two different keys have the same hash. Python can still keep both because it checks equality too.

The custom class below forces a collision on purpose. This is an **advanced illustration**, not something you need to implement for a basic slicing/dictionary exam.


In [ ]:
class CollisionExample:
    def __init__(self, label):
        self.label = label

    def __hash__(self):
        return 7                  # Deliberately give every instance the same hash.

    def __eq__(self, other):
        return isinstance(other, CollisionExample) and self.label == other.label

    def __repr__(self):
        return f"CollisionExample({self.label!r})"

alpha = CollisionExample("alpha")
beta = CollisionExample("beta")
collision_dict = {alpha: "first", beta: "second"}

print("Same hash?", hash(alpha) == hash(beta))
print("Equal objects?", alpha == beta)
print("Dictionary:", collision_dict)
assert hash(alpha) == hash(beta) and alpha != beta and len(collision_dict) == 2


## 8. Practice: predict, then run

For each object, predict **hashable or not** before running. Try changing one element inside the nested tuples.

A reliable test is to actually call `hash()`. A type-level test like `isinstance(value, collections.abc.Hashable)` cannot always detect an unhashable item hidden *inside a tuple*.


In [ ]:
def is_hashable(value):
    try:
        hash(value)
        return True
    except TypeError:
        return False

quiz = [
    ((1, "x"), True),
    ((1, [2]), False),
    (frozenset({1, 2}), True),
    ([1, 2], False),
    ({"key": "value"}, False),
    ("123", True),
]
for value, expected in quiz:
    actual = is_hashable(value)
    print(f"{value!r:<28} predicted answer: {expected!s:<5} result: {actual}")
    assert actual == expected
print("\nAll six answer checks passed. Add a seventh example!")


## Key takeaways

1. **Hashable does not mean string:** many data types can be dictionary keys.
2. `hash(value)` returns an integer for a hashable value, and its hash must remain stable during its lifetime.
3. `list`, `dict`, and `set` are unhashable; a `tuple` is hashable only if its contents are hashable.
4. Equal keys need equal hashes. Different keys can share a hash without being equal.
5. **Exam trap:** `True`, `1`, and `1.0` refer to the same dictionary key.
6. To check a particular value, call `hash()` and catch `TypeError`.

**References:** [Python glossary: hashable](https://docs.python.org/3/glossary.html#term-hashable) · [Python: `hash()`](https://docs.python.org/3/library/functions.html#hash) · [Python: mapping types (`dict`)](https://docs.python.org/3/library/stdtypes.html#mapping-types-dict)
